# Phase 2 - Diagnosis head  —  Rotax_914_ULF

Which of the 8 sensor channels is faulted, and with which of the 6 fault types
(none/Bias/Drift/Spike/Stuck-At/Noise). One-of-N **per channel**.

In [1]:
import os, sys, json, time
sys.path.insert(0, os.path.abspath("."))
import numpy as np, tensorflow as tf
from tensorflow import keras
import tf_data_pipeline as P, model_architectures as A, train_common as C

C.report_gpu()

  TF 2.16.2  devices ['CPU', 'GPU']


['CPU', 'GPU']

In [2]:
# ---- SET THE ENGINE HERE. Run this notebook once per engine. ----
ENGINE = "Rotax_914_ULF"
steps = C.describe(ENGINE)
train_ds, val_ds, test_ds = C.datasets(ENGINE)

Rotax_914_ULF  (914)
  physics   : v3   TBO 2000.0 h
  rows      : 10,001,902 in 1214 scenarios
  steps/epoch: train 982  val 94  test 130
  window 128  stride 64  features 25  aux 10


2026-09-12 16:05:41.538340: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M2
2026-09-12 16:05:41.538702: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 8.00 GB
2026-09-12 16:05:41.538723: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 2.67 GB
2026-09-12 16:05:41.539610: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-09-12 16:05:41.539633: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


### Class weighting
`none` dominates so heavily that plain inverse-frequency weighting pushes the head
to predict faults everywhere. Inverse-**sqrt** frequency, capped, computed on train only.

In [3]:
xi  = A.build_encoder_input()
enc = A.build_encoder(xi)
det = A.build_detection_head(enc)
dia = A.build_diagnosis_head(enc)
model = keras.Model(xi, {"y_detection": det, "y_diagnosis": dia}, name="diagnosis")
model.summary()

Model: "diagnosis"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ x (InputLayer)      │ (None, 128, 25)   │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn0_conv1 (Conv1D) │ (None, 128, 48)   │      3,648 │ x[0][0]           │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn0_drop1          │ (None, 128, 48)   │          0 │ tcn0_conv1[0][0]  │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn0_conv2 (Conv1D) │ (None, 128, 48)   │      6,960 │ tcn0_drop1[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn0_drop2          │ (None, 128, 48)   │          0 │ tcn0_conv2[0][0]  │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn0_proj (Conv1D)  │ (None, 128, 48)   │      1,248 │ x[0][0]           │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn0_add (Add)      │ (None, 128, 48)   │          0 │ tcn0_drop2[0][0], │
│                     │                   │            │ tcn0_proj[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn0_relu (ReLU)    │ (None, 128, 48)   │          0 │ tcn0_add[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn1_conv1 (Conv1D) │ (None, 128, 48)   │      6,960 │ tcn0_relu[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn1_drop1          │ (None, 128, 48)   │          0 │ tcn1_conv1[0][0]  │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn1_conv2 (Conv1D) │ (None, 128, 48)   │      6,960 │ tcn1_drop1[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn1_drop2          │ (None, 128, 48)   │          0 │ tcn1_conv2[0][0]  │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn1_add (Add)      │ (None, 128, 48)   │          0 │ tcn1_drop2[0][0], │
│                     │                   │            │ tcn0_relu[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn1_relu (ReLU)    │ (None, 128, 48)   │          0 │ tcn1_add[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn2_conv1 (Conv1D) │ (None, 128, 48)   │      6,960 │ tcn1_relu[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn2_drop1          │ (None, 128, 48)   │          0 │ tcn2_conv1[0][0]  │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn2_conv2 (Conv1D) │ (None, 128, 48)   │      6,960 │ tcn2_drop1[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn2_drop2          │ (None, 128, 48)   │          0 │ tcn2_conv2[0][0]  │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn2_add (Add)      │ (None, 128, 48)   │          0 │ tcn2_drop2[0][0], │
│                     │                   │            │ tcn1_relu[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn2_relu (ReLU)    │ (None, 128, 48)   │          0 │ tcn2_add[0][0]    │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 98,529 (384.88 KB)

 Trainable params: 98,529 (384.88 KB)

 Non-trainable params: 0 (0.00 B)

In [4]:
cw = C.diagnosis_class_weights(ENGINE)
np.save(os.path.join(C.MODELS_DIR, f"diag_class_weights_{C.ENGINE_KEY[ENGINE]}.npy"), cw)
print("class weights:", cw.round(3))

class weights: [ 1.    10.     6.184  9.996  6.402  3.537]


In [5]:
model.compile(optimizer=keras.optimizers.Adam(1e-3, clipnorm=1.0),

              loss={"y_detection": keras.losses.BinaryCrossentropy(),

                    "y_diagnosis": C.weighted_diagnosis_loss(cw)},

              loss_weights={"y_detection": 1.0, "y_diagnosis": 50.0},
              metrics={"y_diagnosis": [C.DiagnosisMacroF1(), C.DiagnosisFaultRecall()]})

In [6]:
PHASE = "phase2_diagnosis"

PREV  = "phase1_detection"

ckpt  = C.ckpt_path(ENGINE, PHASE)

if PREV: C.warm_start(model, C.ckpt_path(ENGINE, PREV))



hist = model.fit(

    train_ds.map(C.split_labels(["y_detection","y_diagnosis"])),

    validation_data=val_ds.map(C.split_labels(["y_detection","y_diagnosis"])),

    steps_per_epoch=steps["train"],

    validation_steps=steps["val"],

    epochs=20,

    callbacks=C.callbacks(ckpt, monitor="val_y_diagnosis_macro_f1", mode="max"),

    verbose=1,

)

print("saved:", ckpt)

  warm-started 1 layers from phase1_detection_914.keras
Epoch 1/20


/Users/harsh/Documents/UAV_Engine/validation/venv/lib/python3.11/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
2026-09-12 16:05:49.202883: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


982/982 ━━━━━━━━━━━━━━━━━━━━ 0s 216ms/step - loss: 31.3768 - y_detection_loss: 0.5017 - y_diagnosis_loss: 0.6175
Epoch 1: val_loss improved from None to 19.27337, saving model to /Users/harsh/Documents/UAV_Engine/validation/models/phase2_diagnosis_914.keras

Epoch 1: finished saving model to /Users/harsh/Documents/UAV_Engine/validation/models/phase2_diagnosis_914.keras
982/982 ━━━━━━━━━━━━━━━━━━━━ 228s 224ms/step - loss: 31.3768 - y_detection_loss: 0.5017 - y_diagnosis_loss: 0.6175 - val_loss: 19.2734 - val_y_detection_loss: 0.3785 - val_y_diagnosis_loss: 0.3779 - learning_rate: 0.0010
Epoch 2/20
872/982 ━━━━━━━━━━━━━━━━━━━━ 24s 222ms/step - loss: 103.4930 - y_detection_loss: 2.9496 - y_diagnosis_loss: 2.0109

KeyboardInterrupt: 

In [ ]:
res = model.evaluate(test_ds.map(C.split_labels(["y_detection","y_diagnosis"])), steps=steps["test"], verbose=1, return_dict=True)
print(json.dumps({k: float(v) for k, v in res.items()}, indent=2))
with open(os.path.join(C.MODELS_DIR, f"{PHASE}_{C.ENGINE_KEY[ENGINE]}_test.json"), "w") as f:
    json.dump({k: float(v) for k, v in res.items()}, f, indent=2)